<a href="https://colab.research.google.com/github/tony-97/idat-sync/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -U git+https://github.com/tony-97/idat-sync moviepy
!playwright install chromium  --with-deps
!apt install weasyprint
!apt-get install xattr

  Cloning https://github.com/tony-97/idat-sync to /tmp/pip-req-build-7dk2psun
  Running command git clone --filter=blob:none --quiet https://github.com/tony-97/idat-sync /tmp/pip-req-build-7dk2psun
  Resolved https://github.com/tony-97/idat-sync to commit e097c84289828c89ee310b247b16bafeba1fe4d7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/vgrem/office365-rest-python-client.git (to revision 8268769c2575549a6e8af87c5ba4518f3a714cf3) to /tmp/pip-install-5gpkmwnb/office365-rest-python-client_1752711aed3d44669767aa75b1fc180b
  Running command git clone --filter=blob:none --quiet https://github.com/vgrem/office365-rest-python-client.git /tmp/pip-install-5gpkmwnb/office365-rest-python-client_1752711aed3d44669767aa75b1fc180b
  Running command git rev-parse -q --verify 'sha^8268769c2575549a6e8af87c5ba4518f3a714cf3'
  Running command git fetch -q https://github.com/vgrem/office

In [7]:
from google.colab import drive, userdata, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

from idat_sync.idat_sync import IDATSync, get_course_contents, call_ws
from idat_sync.auth import AuthProvider
from idat_sync.utils import safe_filename

from moviepy import VideoFileClip

from weasyprint import HTML

import os
import asyncio
import mimetypes
from pathlib import Path
from subprocess import getoutput


def get_share_link(path):
  file_id = getoutput(f'xattr -p "user.drive.id" "{path}"')
  return f"https://drive.google.com/file/d/{file_id}"

def html_to_pdf(path):
  filename = os.path.splitext(path)[0]
  output_path = f"{filename}.pdf"
  HTML(filename=path).write_pdf(output_path)
  print(f"Successfully converted {path} to {output_path}")
  return output_path

def video_to_mp3(path):
  filename = os.path.splitext(path)[0]
  output_audio_path = f"{filename}.mp3"
  # Load the video file
  video_clip = VideoFileClip(path)

  # Extract the audio
  audio_clip = video_clip.audio

  # Write the audio file
  audio_clip.write_audiofile(output_audio_path, logger=None)

  # Close the clips
  audio_clip.close()
  video_clip.close()
  print(f"Successfully extracted audio to {output_audio_path}")
  return output_audio_path

def print_courses(courses):
    print("\nYour Moodle courses:")
    for i, c in enumerate(courses, 1):
        fullname = c.get("fullname") or c.get("shortname")
        print(f"{i:>2}. [{c.get('id')}] {fullname}")


def choose_course(courses):
    while True:
        s = input("\nSelect course number: ").strip()
        if not s.isdigit():
            print("Enter a number from the list.")
            continue
        idx = int(s)
        if 1 <= idx <= len(courses):
            return courses[idx - 1]
        print("Out of range.")

async def get_mfa_token():
    code = await asyncio.to_thread(input, ">> CONSOLE: Enter 6-digit MFA code: ")
    return code

drive.mount('/content/drive', force_remount=True)


auth = AuthProvider()
login_flow_ended = asyncio.Event()


#await auth.login("iv71430260", userdata.get("PASSWORD"), login_flow_ended, get_mfa_token)
credentials = auth.get_credentials()

idat_sync = IDATSync(credentials)
courses = idat_sync.list_my_courses()
print_courses(courses)
course = choose_course(courses)
assignments_courses = call_ws(idat_sync.token, "mod_assign_get_assignments").get(
            "courses", []
)

if (cid := course.get("id")) and (course_name := course.get("fullname")):
  print(
      f"\nFetching contents for: {course.get('fullname') or course.get('shortname')} (id={cid}) ..."
  )
  contents = get_course_contents(idat_sync.token, cid)
  if isinstance(contents, dict) and contents.get("exception"):
      print("[!] Error:", contents)
      exit(1)
  assignments = (
      next(
          (
              course_assignment
              for course_assignment in assignments_courses
              if course_assignment.get("id") == cid
          ),
          {},
      )
  ).get("assignments", [])
  course_folder = os.path.join("drive/MyDrive/sync", safe_filename(course_name))
  #idat_sync.sync_course(contents, assignments, course_name, course_folder)
  links = []
  recordings = []
  for root, dirs, files in os.walk(course_folder):
    for file in files:
      path = os.path.join(root, file)
      filename, ext = os.path.splitext(path)
      mime_type, _ = mimetypes.guess_type(path)
      if ext == ".html":
        if os.path.exists(f"{filename}.pdf"):
          continue
        path = html_to_pdf(path)
      elif ext == ".mp3":
        recordings.append(path)
        continue
      elif mime_type and mime_type.startswith("video/"):
        if os.path.exists(f"{filename}.mp3"):
          continue
        mp3 = video_to_mp3(path)
        recordings.append(mp3)
        continue
      share_link = get_share_link(path)
      links.append(share_link)

  recordings = list(map(lambda path: get_share_link(path), recordings))
  print("=========Recordings=======")
  for recording in recordings:
    print(recording)
  print("=========Links=======")
  for link in links:
    print(link)



Mounted at /content/drive

Your Moodle courses:
 1. [26553] I.01.2025-IIA DESARROLLO DE HABILIDADES INTRAPERSONALES
 2. [26551] I.01.2025-IIA FUNDAMENTOS LÓGICO-MATEMÁTICOS PARA EL DESARROLLO DE SISTEMAS
 3. [26550] I.01.2025-IIA LECTURA COMPRENSIVA Y ESCRITURA
 4. [26549] I.01.2025-IIA SOPORTE DE TI
 5. [26548] I.01.2025-IIA DESARROLLO WEB
 6. [26547] I.01.2025-IIA FUNDAMENTOS DE PROGRAMACIÓN
 7. [27005] I.03.2025-IIA HERRAMIENTAS OFIMÁTICAS PARA LA INVESTIGACIÓN
 8. [35379] III.01.2026-I SEMINARIO SOBRE CREATIVIDAD E INNOVACIÓN
 9. [33930] III.01.2026-I REDES INFORMÁTICAS
10. [33929] III.01.2026-I ANÁLISIS Y DISEÑO DE SISTEMAS ORIENTADOS A OBJETOS
11. [33928] III.01.2026-I DESARROLLO DE INTERFACES 2
12. [33927] III.01.2026-I PROGRAMACIÓN DE BASE DE DATOS
13. [33926] III.01.2026-I INVESTIGACIÓN ÉTICA Y RESPONSABILIDAD CIUDADANA
14. [33925] III.01.2026-I DESARROLLO PROFESIONAL
15. [32500] II.01.2025-III SEMINARIO SOBRE TECNOLOGÍA E INTELIGENCIA ARTIFICIAL
16. [31568] II.01.2025-III EFS